# FINE + Morlet Scalogram Front-End (Technique 2) - local

Interactive driver for **one pair at a time**. The pipeline lives in `fine_mi.py`;
this notebook only calls it, so it can never drift from the batch runner.

For the full sweep use the command line instead - it is resumable and unattended:

```
python run_local.py --selftest                  # checks only, no training
python run_local.py --mode cwt --pairs HOC/SPS --smoke
python run_local.py --mode both --pairs all --time-probe
python run_local.py --mode raw --pairs all
python run_local.py --mode cwt --pairs all
python aggregate.py
```

**Two arms, one code path.** `raw` = cue-aligned EEG into FINE-1D (baseline);
`cwt` = Morlet scalograms into FINE-2D (Technique 2). Both share fold indices, seeds,
normalisation and augmentation draws, diverging only inside `apply_frontend`.

**Differences from `EMBC_deterministic-3.ipynb` (all deliberate):**

| | old | new |
|---|---|---|
| window slice | `X[:, :, :t]` (starts 500 ms pre-cue) | `X[:, :, 125:1125]` (cue-aligned) |
| windows | 800 / 1500 / 3000 / 4000 ms | 4000 ms only |
| front-end | none | `raw` / `cwt` |
| pooling | `AdaptiveAvgPool1d` | `.mean(dim=...)` - 2D adaptive pools are nondeterministic |
| loader | globs `subject*/` directories | globs `subject*.npz` files |
| results | `mi_results/<PAIR>/` | `mi_results_v2/<mode>/<PAIR>/` |

In [ ]:
# fine_mi sets CUBLAS_WORKSPACE_CONFIG before importing torch, so it must be imported
# FIRST on a fresh kernel. Restart the kernel if anything else already pulled in torch.
import importlib
import numpy as np
import fine_mi as F
importlib.reload(F)

device, GPU = F.setup_determinism()
print(F.describe_environment(device, GPU))
print(f"\ndataset: {F.DATASET_ROOT}")
print(f"results: {F.RESULTS_ROOT}")

In [ ]:
# ===========================================================================
# CONFIG - the only things you edit between runs
# ===========================================================================
FEATURE_MODE = 'cwt'      # 'raw' | 'cwt'
CLASS_A, CLASS_B = 0, 5   # see F.JOINT_NAMES
SMOKE = True              # True => 2 subjects, 5 epochs, nothing saved

N_EPOCHS = 5 if SMOKE else F.N_EPOCHS
MAX_SUBJECTS = 2 if SMOKE else None

print(f"MODE : {FEATURE_MODE}")
print(f"PAIR : {F.pair_name(CLASS_A, CLASS_B)}  (classes {CLASS_A} vs {CLASS_B})")
print(f"WIN  : {F.WINDOW_MS} ms = {F.WINDOW_SAMPLES} samples, cue-aligned "
      f"[{F.CUE_SAMPLE}:{F.CUE_SAMPLE + F.WINDOW_SAMPLES}]")
print(f"CWT  : {F.N_FREQS} freqs {F.FREQS[0]:.1f}-{F.FREQS[-1]:.1f} Hz, "
      f"n_cycles {F.N_CYCLES[0]:.0f}->{F.N_CYCLES[-1]:.0f}, decim {F.TIME_DECIM} "
      f"-> (C, {F.N_FREQS}, {F.T_OUT})")
print(f"kernel K={F.CWT_K}, pad={F.CWT_PAD} | context [{F.CTX_START}:{F.CTX_END}]")
if SMOKE:
    print("\nSMOKE MODE - 2 subjects, 5 epochs, nothing is saved")

In [ ]:
# ===========================================================================
# SELF-TESTS - run once on a new machine before committing to a long sweep
# (identical to `python run_local.py --selftest`)
# ===========================================================================
import subprocess, sys
print(subprocess.run([sys.executable, "run_local.py", "--selftest"],
                     capture_output=True, text=True).stdout)

In [ ]:
# ===========================================================================
# RUN ONE PAIR
# ===========================================================================
results = F.run_pair(CLASS_A, CLASS_B, FEATURE_MODE, device,
                     n_epochs=N_EPOCHS, max_subjects=MAX_SUBJECTS)

In [ ]:
# ===========================================================================
# SAVE
# ===========================================================================
if SMOKE:
    accs = [r['time_window_results'][F.WINDOW_MS]['test_accuracy'] for r in results]
    print(f"SMOKE MODE - not saving. {len(results)} subjects, mean {np.mean(accs):.2f}%")
else:
    pair_dir, wide = F.save_results(results, CLASS_A, CLASS_B, FEATURE_MODE, GPU)

---
## Aggregation

Run once both arms have covered the pairs you care about. Independent of everything above,
and identical to `python aggregate.py`.

In [ ]:
import aggregate
importlib.reload(aggregate)

tab = aggregate.build_table(F.RESULTS_ROOT)
if tab is not None:
    display(tab)